In [1]:
from pyspark import SparkConf, SparkContext
conf = SparkConf().setAppName("Prova_esame")
sc = SparkContext(conf=conf)

#Task 1

In [9]:
outputPath1="./output1/"
actionsRDD=sc.textFile("./data/Actions.txt")
appsRDD=sc.textFile("./data/Apps.txt")

In [25]:
cleanedActionsRDD=actionsRDD.map(lambda x: (x.split(",")[0],(x.split(",")[1],x.split(",")[2].split("/")[0],x.split(",")[3]))) \
    .filter(lambda x: x[1][1]=='2022' and x[1][2]=='Install').distinct()\
    .map(lambda x: (x[1][0],x[0]))

cleanedApps=appsRDD.map(lambda x: (x.split(",")[0],float(x.split(",")[2])))
joinedRDD=cleanedActionsRDD.join(cleanedApps).map(lambda x: (x[1][0],(1,0)) if x[1][1]==0 else (x[1][0],(0,1))) \
    .reduceByKey(lambda a,b: (a[0]+b[0],a[1]+b[1])).filter(lambda x: x[1][1]>x[1][0]).map(lambda x: (x[0],x[1][1]))

#Task 2

In [28]:
usersRDD=sc.textFile("./data/Users.txt")
outputPath1="./output2/"

In [33]:
cleanedUsersRDD=usersRDD.map(lambda x: (x.split(",")[0],x.split(",")[3])).filter(lambda x: x[1]=='Italian')

def filtraggio(row1, row2):
    date1=row1[0]
    date2=row2[0]
    state1=row1[1]
    state2=row2[1]
    if (date1>date2):
      return (date1,state1)
    else:
      return (date2,state2)


newCleanedActionsRDD=actionsRDD.map(lambda x: ((x.split(",")[0],x.split(",")[1]),(x.split(",")[2],x.split(",")[3]))) \
    .reduceByKey(filtraggio).map(lambda x: (x[0][0],1) if x[1][1]=='Install' else (x[0][0],0)) \
    .reduceByKey(lambda a,b: a+b)

maxInstalled=newCleanedActionsRDD.values().max()

finalRDD=newCleanedActionsRDD.filter(lambda x: x[1]==maxInstalled).join(cleanedUsersRDD).map(lambda x: x[0])

In [34]:
finalRDD.collect()

['User1']